In [3]:
import pandas as pd
import numpy as np
import joblib

df = pd.read_csv("../data/model_ready/model_ready_data.csv")

model = joblib.load("../models/lead_time_model.pkl")

In [4]:
features = [

    'Sales',
    'Units',
    'Gross Profit',
    'Cost',
    'Profit_Margin',
    'Sales_Per_Unit',
    'Profit_Per_Unit',
    'Factory_Load',
    'Product_Demand',
    'Regional_Demand',
    'Route_Volume',
    'Factory_Efficiency',
    'Factory_Encoded',
    'Region_Encoded',
    'ShipMode_Encoded',
    'Product_Encoded',
    'Shipping_Distance_KM',
    'Distance_Efficiency',
    'Factory_Avg_Distance',
    'Route_Risk_Score',
    'Distance_Profit_Ratio',
    'Route_Performance_Index'
]

In [5]:
selected_product = "Laffy Taffy"

product_df = df[
    df["Product Name"] == selected_product
].copy()

In [6]:
current_factory = product_df["Factory"].iloc[0]

print(current_factory)

Sugar Shack


In [7]:
factories = df["Factory"].unique()

factories

<StringArray>
[  'Wicked Choccy's',     'Lot's O' Nuts',    'Secret Factory',
 'The Other Factory',       'Sugar Shack']
Length: 5, dtype: str

In [8]:
factory_encoding = {

    factory: idx

    for idx, factory

    in enumerate(
        sorted(
            df["Factory"].unique()
        )
    )
}

In [9]:
factory_encoding

{"Lot's O' Nuts": 0,
 'Secret Factory': 1,
 'Sugar Shack': 2,
 'The Other Factory': 3,
 "Wicked Choccy's": 4}

In [10]:
results = []

for factory in factories:

    scenario = product_df.copy()

    scenario["Factory"] = factory

    scenario["Factory_Encoded"] = (
        factory_encoding[factory]
    )

    X_sim = scenario[features]

    preds = model.predict(X_sim)

    avg_lt = preds.mean()

    results.append({

        "Factory": factory,

        "Predicted_Lead_Time": avg_lt

    })

In [11]:
simulation_results = pd.DataFrame(results)

simulation_results

,Factory,Predicted_Lead_Time
0,Wicked Choccy's,1384.280
1,Lot's O' Nuts,1385.759
2,Secret Factory,1385.759
3,The Other Factory,1384.288
4,Sugar Shack,1384.290


In [12]:
simulation_results = (
    simulation_results
    .sort_values(
        by="Predicted_Lead_Time"
    )
)

simulation_results

,Factory,Predicted_Lead_Time
0,Wicked Choccy's,1384.280
3,The Other Factory,1384.288
4,Sugar Shack,1384.290
1,Lot's O' Nuts,1385.759
2,Secret Factory,1385.759


In [13]:
current_lt = simulation_results[
    simulation_results["Factory"]
    ==
    current_factory
]["Predicted_Lead_Time"].values[0]

In [14]:
simulation_results[
    "Improvement_%"
] = (

    (
        current_lt
        -
        simulation_results[
            "Predicted_Lead_Time"
        ]
    )

    /

    current_lt

) * 100

In [15]:
simulation_results

,Factory,Predicted_Lead_Time,Improvement_%
0,Wicked Choccy's,1384.280,0.000722
3,The Other Factory,1384.288,0.000144
4,Sugar Shack,1384.290,0.000000
1,Lot's O' Nuts,1385.759,-0.106119
2,Secret Factory,1385.759,-0.106119


In [16]:
best_factory = simulation_results.iloc[0]

best_factory

Factory                Wicked Choccy's
Predicted_Lead_Time            1384.28
Improvement_%                 0.000722
Name: 0, dtype: object

In [17]:
simulation_results.to_csv(

    "../outputs/recommendations.csv",

    index=False
)

In [18]:
simulation_results

,Factory,Predicted_Lead_Time,Improvement_%
0,Wicked Choccy's,1384.280,0.000722
3,The Other Factory,1384.288,0.000144
4,Sugar Shack,1384.290,0.000000
1,Lot's O' Nuts,1385.759,-0.106119
2,Secret Factory,1385.759,-0.106119


In [21]:
import pandas as pd
import numpy as np
import joblib

In [22]:
df = pd.read_csv("../data/model_ready/model_ready_data.csv")

model = joblib.load("../models/lead_time_model.pkl")

# Features list 

In [23]:
features = [
    'Sales',
    'Units',
    'Gross Profit',
    'Cost',
    'Profit_Margin',
    'Sales_Per_Unit',
    'Profit_Per_Unit',
    'Factory_Load',
    'Product_Demand',
    'Regional_Demand',
    'Route_Volume',
    'Factory_Efficiency',
    'Factory_Encoded',
    'Region_Encoded',
    'ShipMode_Encoded',
    'Product_Encoded',
    'Shipping_Distance_KM',
    'Distance_Efficiency',
    'Factory_Avg_Distance',
    'Route_Risk_Score',
    'Distance_Profit_Ratio',
    'Route_Performance_Index'
]

# Factory encoding 

In [24]:
factory_encoding = {
    factory: idx
    for idx, factory
    in enumerate(
        sorted(
            df["Factory"].unique()
        )
    )
}

# product list 

In [25]:
products = df["Product Name"].unique()

len(products)

15

# Recommendation engine 

In [26]:
final_results = []

# loop through product

In [27]:
for product in products:

    product_df = df[
        df["Product Name"] == product
    ].copy()

    current_factory = (
        product_df["Factory"]
        .iloc[0]
    )

    simulation_results = []

    for factory in df["Factory"].unique():

        scenario = product_df.copy()

        scenario["Factory"] = factory

        scenario["Factory_Encoded"] = (
            factory_encoding[factory]
        )

        X_sim = scenario[features]

        preds = model.predict(X_sim)

        avg_lt = preds.mean()

        simulation_results.append({

            "Factory": factory,

            "Predicted_Lead_Time": avg_lt

        })

    sim_df = pd.DataFrame(
        simulation_results
    )

    sim_df = sim_df.sort_values(
        by="Predicted_Lead_Time"
    )

    current_lt = sim_df[
        sim_df["Factory"]
        ==
        current_factory
    ]["Predicted_Lead_Time"].values[0]

    best_factory = sim_df.iloc[0]

    improvement = (

        (
            current_lt
            -
            best_factory[
                "Predicted_Lead_Time"
            ]
        )

        /

        current_lt

    ) * 100

    final_results.append({

        "Product": product,

        "Current_Factory":
        current_factory,

        "Recommended_Factory":
        best_factory["Factory"],

        "Current_Lead_Time":
        current_lt,

        "Recommended_Lead_Time":
        best_factory[
            "Predicted_Lead_Time"
        ],

        "Improvement_%":
        improvement
    })

# Final Recommendation

In [28]:
recommendations = pd.DataFrame(
    final_results
)

recommendations

,Product,Current_Factory,Recommended_Factory,Current_Lead_Time,Recommended_Lead_Time,Improvement_%
0,Wonka Bar - Milk Chocolate,Wicked Choccy's,The Other Factory,1316.763304,1316.436448,0.024823
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,The Other Factory,1325.658998,1325.309816,0.026340
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Sugar Shack,1328.918928,1326.328657,0.194916
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,The Other Factory,1320.641613,1317.690470,0.223463
4,Wonka Bar - Fudge Mallows,Lot's O' Nuts,The Other Factory,1314.246320,1311.612778,0.200384
5,Wonka Gum,Secret Factory,Sugar Shack,1305.536167,1303.337333,0.168424
6,Kazookles,The Other Factory,The Other Factory,1268.862500,1268.862500,0.000000
7,Lickable Wallpaper,Secret Factory,Sugar Shack,1338.291596,1336.108191,0.163149
8,Fizzy Lifting Drinks,Sugar Shack,Wicked Choccy's,1263.630000,1263.571667,0.004616
9,Laffy Taffy,Sugar Shack,Wicked Choccy's,1384.290000,1384.280000,0.000722


# Sort by Improvement

In [29]:
recommendations = (
    recommendations
    .sort_values(
        by="Improvement_%",
        ascending=False
    )
)

recommendations

,Product,Current_Factory,Recommended_Factory,Current_Lead_Time,Recommended_Lead_Time,Improvement_%
13,Everlasting Gobstopper,Secret Factory,Sugar Shack,1405.503333,1395.693333,0.697971
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,The Other Factory,1320.641613,1317.690470,0.223463
4,Wonka Bar - Fudge Mallows,Lot's O' Nuts,The Other Factory,1314.246320,1311.612778,0.200384
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Sugar Shack,1328.918928,1326.328657,0.194916
5,Wonka Gum,Secret Factory,Sugar Shack,1305.536167,1303.337333,0.168424
7,Lickable Wallpaper,Secret Factory,Sugar Shack,1338.291596,1336.108191,0.163149
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,The Other Factory,1325.658998,1325.309816,0.026340
0,Wonka Bar - Milk Chocolate,Wicked Choccy's,The Other Factory,1316.763304,1316.436448,0.024823
8,Fizzy Lifting Drinks,Sugar Shack,Wicked Choccy's,1263.630000,1263.571667,0.004616
10,SweeTARTS,Sugar Shack,Wicked Choccy's,1352.307000,1352.250000,0.004215


# Top recommendation

In [30]:
recommendations.head(10)

,Product,Current_Factory,Recommended_Factory,Current_Lead_Time,Recommended_Lead_Time,Improvement_%
13,Everlasting Gobstopper,Secret Factory,Sugar Shack,1405.503333,1395.693333,0.697971
3,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,The Other Factory,1320.641613,1317.690470,0.223463
4,Wonka Bar - Fudge Mallows,Lot's O' Nuts,The Other Factory,1314.246320,1311.612778,0.200384
2,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Sugar Shack,1328.918928,1326.328657,0.194916
5,Wonka Gum,Secret Factory,Sugar Shack,1305.536167,1303.337333,0.168424
7,Lickable Wallpaper,Secret Factory,Sugar Shack,1338.291596,1336.108191,0.163149
1,Wonka Bar - Triple Dazzle Caramel,Wicked Choccy's,The Other Factory,1325.658998,1325.309816,0.026340
0,Wonka Bar - Milk Chocolate,Wicked Choccy's,The Other Factory,1316.763304,1316.436448,0.024823
8,Fizzy Lifting Drinks,Sugar Shack,Wicked Choccy's,1263.630000,1263.571667,0.004616
10,SweeTARTS,Sugar Shack,Wicked Choccy's,1352.307000,1352.250000,0.004215


# Save

In [31]:
recommendations.to_csv(
    "../outputs/final_recommendations.csv",
    index=False
)

# Executive Metrics 

In [32]:
(
recommendations[
recommendations[
    "Improvement_%"
] > 0
]
.shape[0]
)


14

In [33]:
recommendations[
    "Improvement_%"
].mean()


np.float64(0.11420477940813877)

In [34]:
recommendations.iloc[0]

Product                  Everlasting Gobstopper
Current_Factory                  Secret Factory
Recommended_Factory                 Sugar Shack
Current_Lead_Time                   1405.503333
Recommended_Lead_Time               1395.693333
Improvement_%                          0.697971
Name: 13, dtype: object

# Checking Model Existence

In [35]:
import os

os.path.exists(
"../models/lead_time_model.pkl"
)

True

In [36]:
import pandas as pd

df = pd.read_csv(
"../outputs/final_recommendations.csv"
)

df.head()

,Product,Current_Factory,Recommended_Factory,Current_Lead_Time,Recommended_Lead_Time,Improvement_%
0,Everlasting Gobstopper,Secret Factory,Sugar Shack,1405.503333,1395.693333,0.697971
1,Wonka Bar -Scrumdiddlyumptious,Lot's O' Nuts,The Other Factory,1320.641613,1317.690470,0.223463
2,Wonka Bar - Fudge Mallows,Lot's O' Nuts,The Other Factory,1314.246320,1311.612778,0.200384
3,Wonka Bar - Nutty Crunch Surprise,Lot's O' Nuts,Sugar Shack,1328.918928,1326.328657,0.194916
4,Wonka Gum,Secret Factory,Sugar Shack,1305.536167,1303.337333,0.168424


In [37]:
df.shape

(15, 6)